In [ ]:
# 实验5：阶段1 环境检查
import sys, subprocess, json, random
from pathlib import Path
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.1', 'pillow', 'matplotlib', 'pandas'])
import torch, transformers
random.seed(42); torch.manual_seed(42)
OUT=Path('/content/vqa_outputs'); OUT.mkdir(exist_ok=True)
print('Python:',sys.version,'Transformers:',transformers.__version__)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), '请连接 GPU 后重试'
# 实验5：环境检查
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pillow','matplotlib','pandas'])
import torch
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# 阶段2：BLIP 预训练基线与演示问答（不是正式准确率评测）
from pathlib import Path
import json, time, requests, torch
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
from transformers import BlipProcessor, BlipForQuestionAnswering
OUT=Path('/content/vqa_outputs'); OUT.mkdir(exist_ok=True)
MODEL_ID='Salesforce/blip-vqa-base'
device='cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
processor=BlipProcessor.from_pretrained(MODEL_ID)
model=BlipForQuestionAnswering.from_pretrained(MODEL_ID).to(device).eval()
url='https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg'
r=requests.get(url,timeout=60); r.raise_for_status()
im=Image.open(BytesIO(r.content)).convert('RGB'); im.save(OUT/'demo.jpg')
questions=['What is the woman doing?','Is there a dog in the picture?','How many dogs are there?']
rows=[]
for question in questions:
    inputs=processor(images=im,text=question,return_tensors='pt').to(device)
    if device=='cuda': torch.cuda.synchronize()
    start=time.perf_counter()
    with torch.inference_mode():
        ids=model.generate(**inputs,max_new_tokens=20,num_beams=1)
    if device=='cuda': torch.cuda.synchronize()
    answer=processor.decode(ids[0],skip_special_tokens=True)
    rows.append({'question':question,'prediction':answer,'seconds':time.perf_counter()-start})
    print(question,'=>',answer)
(OUT/'smoke_predictions.json').write_text(json.dumps(rows,ensure_ascii=False,indent=2))
fig,ax=plt.subplots(figsize=(10,7)); ax.imshow(im); ax.axis('off')
fig.text(.05,.02,'\n'.join(x['question']+' → '+x['prediction'] for x in rows),fontsize=11)
fig.subplots_adjust(bottom=.2); fig.savefig(OUT/'smoke_vqa.png',dpi=200,bbox_inches='tight'); plt.show()
print('仅验证端到端问答；正式 VQAv2 评测尚未执行。输出目录:',OUT)


# 实验5：视觉问答系统

## 已验证的阶段结果
实验3旧运行时已释放。本 notebook 已连接 Tesla T4，Python 3.13.15、Transformers 4.57.1，环境检查通过。

BLIP 演示图片的三条问答已执行成功：
- What is the woman doing? → petting dog
- Is there a dog in the picture? → yes
- How many dogs are there? → 1

代码已输出 smoke_predictions.json、demo.jpg 和 smoke_vqa.png 至 /content/vqa_outputs。演示结果不计入正式准确率。

## 后续实验设计
1. 从 VQAv2 官方 validation 固定种子 42 抽取 500 个问题，保存 question_id、image_id 和十个人工答案。
2. 按官方答案标准化与 leave-one-out 软准确率评估总体及 yes/no、number、other 类别，保留逐题预测。
3. 输出分类准确率、样本分布、推理耗时、成功和失败案例图。
4. 同一评测集合对比 greedy 和 beam search；微调只使用独立 train 子集与开发集。
5. 导出模型版本、配置、预测与图表，用于报告和 PPT。

模型 Salesforce/blip-vqa-base 是经过 VQA 训练的现成基线；本项目尚未进行微调，也尚未完成 VQAv2 正式评测。


In [ ]:
# 阶段3：固定 VQAv2 validation 子集；下载失败不替换样本
import zipfile, hashlib, concurrent.futures, random, json, requests
from pathlib import Path
from PIL import Image
DATA=Path('/content/vqa_data'); DATA.mkdir(exist_ok=True)
IMG=DATA/'images'; IMG.mkdir(exist_ok=True)
OUT=Path('/content/vqa_outputs'); OUT.mkdir(exist_ok=True)
def fetch(url,path):
    if path.exists() and path.stat().st_size>0: return path
    for attempt in range(3):
        try:
            r=requests.get(url,timeout=120); r.raise_for_status()
            tmp=path.with_suffix('.part'); tmp.write_bytes(r.content); tmp.replace(path)
            return path
        except Exception:
            if attempt==2: raise
sources={
'questions':('https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip','v2_OpenEnded_mscoco_val2014_questions.json'),
'annotations':('https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip','v2_mscoco_val2014_annotations.json')}
objects={}; provenance={}
for kind,(url,member) in sources.items():
    p=fetch(url,DATA/(kind+'.zip'))
    with zipfile.ZipFile(p) as z: objects[kind]=json.loads(z.read(member))
    provenance[kind]={'url':url,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()}
    print(kind,'loaded',flush=True)
qs={q['question_id']:q for q in objects['questions']['questions']}
selected=random.Random(42).sample(sorted(objects['annotations']['annotations'],key=lambda a:a['question_id']),500)
manifest=[dict(a,question=qs[a['question_id']]['question']) for a in selected]
(OUT/'eval_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
(OUT/'data_sources.json').write_text(json.dumps(provenance,indent=2))
def image_path(i): return IMG/f'COCO_val2014_{i:012d}.jpg'
def download_image(i):
    p=fetch(f'https://s3.amazonaws.com/images.cocodataset.org/val2014/COCO_val2014_{i:012d}.jpg',image_path(i))
    with Image.open(p) as im: im.verify()
    return i
image_ids=sorted({a['image_id'] for a in manifest}); failures=[]
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
    futures={pool.submit(download_image,i):i for i in image_ids}
    for k,f in enumerate(concurrent.futures.as_completed(futures),1):
        try: f.result()
        except Exception as e: failures.append({'image_id':futures[f],'error':str(e)})
        if k%50==0: print('images',k,'/',len(image_ids),flush=True)
(OUT/'download_failures.json').write_text(json.dumps(failures,indent=2))
print('Questions:',len(manifest),'Images:',len(image_ids),'Failures:',len(failures))
assert not failures, '请重试失败图片；不得悄悄替换评测样本'
import pandas as pd
print(pd.Series([a['answer_type'] for a in manifest]).value_counts())


In [ ]:
# 阶段4：官方 VQA evaluator（仅将两处 Python 2 print 改为 Python 3）
import copy, contextlib, io, importlib.util, hashlib
url='https://raw.githubusercontent.com/GT-Vision-Lab/VQA/master/PythonEvaluationTools/vqaEvaluation/vqaEval.py'
r=requests.get(url,timeout=60); r.raise_for_status(); original=r.text
source=original.replace('print "computing accuracy"','print("computing accuracy")').replace('print "Done computing accuracy"','print("Done computing accuracy")')
(OUT/'vqaEval_original.py.txt').write_text(original)
(OUT/'vqaEval.py').write_text(source)
(OUT/'evaluator_source.json').write_text(json.dumps({'url':url,'sha256_original':hashlib.sha256(r.content).hexdigest(),'modification':'two print statements converted to Python3; scoring unchanged'},indent=2))
spec=importlib.util.spec_from_file_location('official_vqa_eval',OUT/'vqaEval.py'); mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
class VQAView:
    def __init__(self,annotations): self.qa={a['question_id']:a for a in annotations}
    def getQuesIds(self): return list(self.qa)
def official_eval(annotations,predictions):
    evaluator=mod.VQAEval(VQAView(copy.deepcopy(annotations)),VQAView(predictions),n=6)
    with contextlib.redirect_stdout(io.StringIO()): evaluator.evaluate()
    return evaluator
# 核验 10 人答案中有 0/1/2/3/4 个匹配时分别为 0/30/60/90/100 分
for matches,expected in [(0,0),(1,30),(2,60),(3,90),(4,100)]:
    a={'question_id':1,'question_type':'unit','answer_type':'other','answers':[{'answer_id':i+1,'answer':'cat' if i<matches else 'dog'} for i in range(10)]}
    got=official_eval([a],[{'question_id':1,'answer':'cat'}]).accuracy['overall']
    assert abs(got-expected)<1e-6,(matches,got)
print('Official evaluator tests passed: 0,30,60,90,100')


In [ ]:
from pathlib import Path
import json
OUT=Path('/content/vqa_outputs')
print('FILES',[(p.name,p.stat().st_size) for p in OUT.glob('*')])
for label in ['greedy','beam3']:
 p=OUT/f'predictions_{label}.jsonl'
 print(label, len(p.read_text().splitlines()) if p.exists() else 'MISSING')


In [ ]:
# 阶段5：预注册解码对照 greedy / beam3；所有问题均生成答案
import time, platform, transformers, numpy as np, pandas as pd
assert not failures and torch.cuda.is_available()
model.eval(); torch.manual_seed(42)
config={'model':MODEL_ID,'model_commit':getattr(model.config,'_commit_hash',None),'seed':42,'n':500,'split':'VQAv2 val2014 random subset','batch_size':8,'max_new_tokens':20,'beams':[1,3],'dtype':str(next(model.parameters()).dtype),'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'transformers':transformers.__version__,'python':platform.python_version(),'manifest_sha256':hashlib.sha256((OUT/'eval_manifest.json').read_bytes()).hexdigest()}
(OUT/'run_config.json').write_text(json.dumps(config,indent=2))
all_predictions={}
for label,beams in [('greedy',1),('beam3',3)]:
    path=OUT/f'predictions_{label}.jsonl'
    existing=[json.loads(s) for s in path.read_text().splitlines()] if path.exists() else []
    done={p['question_id']:p for p in existing}
    pending=[a for a in manifest if a['question_id'] not in done]
    for start in range(0,len(pending),8):
        batch=pending[start:start+8]
        images=[Image.open(image_path(a['image_id'])).convert('RGB') for a in batch]
        inputs=processor(images=images,text=[a['question'] for a in batch],padding=True,return_tensors='pt').to('cuda')
        torch.cuda.synchronize(); tic=time.perf_counter()
        with torch.inference_mode(): ids=model.generate(**inputs,max_new_tokens=20,num_beams=beams,do_sample=False)
        torch.cuda.synchronize(); elapsed=time.perf_counter()-tic
        answers=processor.batch_decode(ids,skip_special_tokens=True)
        with path.open('a') as f:
            for a,answer in zip(batch,answers):
                row={'question_id':a['question_id'],'image_id':a['image_id'],'question':a['question'],'answer':answer.strip(),'answer_type':a['answer_type'],'question_type':a['question_type'],'batch_seconds_per_question':elapsed/len(batch)}
                f.write(json.dumps(row,ensure_ascii=False)+'\n'); done[a['question_id']]=row
        for im in images: im.close()
        if len(done)%40==0 or len(done)==500: print(label,len(done),'/500',flush=True)
    predictions=[done[a['question_id']] for a in manifest]
    assert len(predictions)==500 and len({x['question_id'] for x in predictions})==500
    all_predictions[label]=predictions
    (OUT/f'submission_{label}.json').write_text(json.dumps([{'question_id':p['question_id'],'answer':p['answer']} for p in predictions]))
    ev=official_eval(manifest,predictions)
    (OUT/f'accuracy_{label}.json').write_text(json.dumps(ev.accuracy,indent=2))
    print(label,ev.accuracy,flush=True)
print('Both configurations complete.')


In [ ]:
# 阶段6：图表、逐题核验、案例选择及可复用报告素材
import json, copy, textwrap, collections, shutil, hashlib, platform
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from IPython.display import display
OUT=Path('/content/vqa_outputs'); FIG=OUT/'figures'; FIG.mkdir(exist_ok=True)
CASE=OUT/'case_images'; CASE.mkdir(exist_ok=True)
manifest=json.loads((OUT/'eval_manifest.json').read_text())
ann={a['question_id']:a for a in manifest}; detail=[]; summaries=[]
for method in ['greedy','beam3']:
    preds=[json.loads(s) for s in (OUT/f'predictions_{method}.jsonl').read_text().splitlines()]
    assert len(preds)==500 and {p['question_id'] for p in preds}==set(ann)
    ev=official_eval(manifest,preds)
    saved=json.loads((OUT/f'accuracy_{method}.json').read_text())
    assert abs(ev.accuracy['overall']-saved['overall'])<1e-6
    for p in preds:
        a=ann[p['question_id']]; counts=collections.Counter(x['answer'] for x in a['answers'])
        detail.append(dict(p,method=method,score=ev.evalQA[p['question_id']],reference_majority=counts.most_common(1)[0][0],references=json.dumps(dict(counts),ensure_ascii=False)))
    for group in ['overall','yes/no','number','other']:
        subset=preds if group=='overall' else [p for p in preds if p['answer_type']==group]
        score=ev.accuracy['overall'] if group=='overall' else ev.accuracy['perAnswerType'][group]
        summaries.append({'method':method,'group':group,'n':len(subset),'accuracy_percent':score})
df=pd.DataFrame(detail); summary=pd.DataFrame(summaries)
df.to_csv(OUT/'per_question_results.csv',index=False); summary.to_csv(OUT/'accuracy_summary.csv',index=False)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.spines.top':False,'axes.spines.right':False,'figure.facecolor':'white'})
colors=['#2463A6','#E38A35']
def savefig(fig,name):
    fig.savefig(FIG/(name+'.png'),dpi=240,bbox_inches='tight')
    fig.savefig(FIG/(name+'.pdf'),bbox_inches='tight'); plt.show(); plt.close(fig)
# 1. 分组准确率；所有纵轴百分比范围固定为 0--100
fig,ax=plt.subplots(figsize=(10,5)); groups=['overall','yes/no','number','other']; x=np.arange(4)
for j,method in enumerate(['greedy','beam3']):
    vals=[summary.query('method==@method and group==@g').accuracy_percent.iloc[0] for g in groups]
    bars=ax.bar(x+(j-.5)*.35,vals,.35,label=method,color=colors[j]); ax.bar_label(bars,fmt='%.2f',padding=3,fontsize=9)
ax.set(xticks=x,xticklabels=['Overall (500)','Yes/No (219)','Number (53)','Other (228)'],ylim=(0,108),ylabel='Official VQA soft accuracy (%)',title='BLIP: fixed 500-question validation diagnostic'); ax.legend(loc='lower right'); ax.grid(axis='y',alpha=.15)
fig.text(.12,-.03,'Potential training overlap: not an unseen-test benchmark.',fontsize=10,color='#8B3030'); savefig(fig,'01_accuracy_by_type')
# 2. 样本数量
fig,ax=plt.subplots(figsize=(7,4)); counts=pd.Series([a['answer_type'] for a in manifest]).value_counts().reindex(['yes/no','number','other'])
bars=ax.bar(counts.index,counts.values,color=['#2463A6','#E38A35','#42958C']); ax.bar_label(bars,padding=3); ax.set(ylabel='Questions',ylim=(0,260),title='Evaluation subset composition (500 questions / 495 images)'); savefig(fig,'02_sample_distribution')
# 3. 摊销批量生成耗时（不含下载、预处理；不能称为单请求延迟）
time_summary=df.groupby('method').batch_seconds_per_question.agg(['mean','median','sum']).reindex(['greedy','beam3']); time_summary.to_csv(OUT/'timing_summary.csv')
fig,ax=plt.subplots(figsize=(7,4)); bars=ax.bar(time_summary.index,1000*time_summary['mean'],color=colors); ax.bar_label(bars,fmt='%.1f',padding=3); ax.set(ylabel='Amortized GPU generation time (ms/question)',title='Generation cost, batch size 8'); ax.margins(y=.2); fig.text(.1,-.04,'Includes CUDA synchronization; excludes image loading and preprocessing.',fontsize=9); savefig(fig,'03_generation_cost')
# 4. 配对变化与图像层面的 bootstrap，不将小幅差异直接当作显著提升
wide=df.pivot(index=['question_id','image_id'],columns='method',values='score'); delta=wide.beam3-wide.greedy
changes=pd.Series({'Improved':int((delta>0).sum()),'Unchanged':int((delta==0).sum()),'Worsened':int((delta<0).sum())})
fig,ax=plt.subplots(figsize=(7,4)); bars=ax.bar(changes.index,changes.values,color=['#42958C','#8993A4','#C45555']); ax.bar_label(bars,padding=3); ax.set(ylabel='Questions',title='Beam3 versus greedy: paired score changes'); ax.margins(y=.15); savefig(fig,'04_paired_changes')
clusters=wide.assign(delta=delta).groupby(level='image_id').agg({'greedy':['sum','count'],'beam3':['sum'],'delta':['sum']})
rng=np.random.default_rng(42); ix=rng.integers(0,len(clusters),(3000,len(clusters))); n=clusters[('greedy','count')].to_numpy()[ix].sum(1)
boot={k:clusters[(k,'sum')].to_numpy()[ix].sum(1)/n for k in ['greedy','beam3','delta']}
ci={k:[float(v) for v in np.percentile(values,[2.5,97.5])] for k,values in boot.items()}
(OUT/'paired_statistics.json').write_text(json.dumps({'delta_beam3_minus_greedy_pp':float(delta.mean()),'changes':changes.to_dict(),'image_cluster_bootstrap_95ci':ci,'resamples':3000,'seed':42},indent=2))
# 5. 有至少 10 个问题的问题类型；小类别不做可靠性排名
qt=df.groupby(['method','question_type']).score.agg(['mean','count']).reset_index(); qt.to_csv(OUT/'question_type_summary.csv',index=False)
qg=qt.query('method=="greedy" and count>=10').sort_values('mean'); fig,ax=plt.subplots(figsize=(9,max(4,len(qg)*.38)))
ax.barh([f'{r.question_type} (n={r.count})' for r in qg.itertuples()],qg['mean'],color=colors[0]); ax.set(xlim=(0,100),xlabel='Official VQA soft accuracy (%)',title='Greedy by question type (n >= 10)'); savefig(fig,'05_question_types')
# 6. 全错、部分得分、满分比例
status=df.assign(outcome=np.select([df.score==0,df.score==100],['Zero credit','Full credit'],default='Partial credit')).groupby(['method','outcome']).size().unstack(fill_value=0).reindex(['greedy','beam3'])
status.to_csv(OUT/'score_outcomes.csv'); fig,ax=plt.subplots(figsize=(8,4)); status[['Zero credit','Partial credit','Full credit']].plot.bar(stacked=True,ax=ax,color=['#C45555','#E7B652','#42958C'],rot=0); ax.set(ylabel='Questions',title='Question-level score outcomes'); ax.legend(loc='upper center',bbox_to_anchor=(.5,-.15),ncol=3); savefig(fig,'06_score_outcomes')
# 按类别、question_id 选取前两个零分案例，避免只挑戏剧性案例
base=df.query('method=="greedy"').sort_values('question_id'); chosen=[]
for kind in ['number','other','yes/no']:
    chosen.extend(base.query('answer_type==@kind and score==0').head(2).question_id.tolist())
case_rows=base[base.question_id.isin(chosen)].copy(); case_rows.to_csv(OUT/'error_cases.csv',index=False)
def gallery(ids,name,title):
    fig,axes=plt.subplots(3,2,figsize=(13,15)); axes=axes.ravel()
    for ax,qid in zip(axes,ids):
        a=ann[qid]; g=df.query('method=="greedy" and question_id==@qid').iloc[0]; b=df.query('method=="beam3" and question_id==@qid').iloc[0]
        path=image_path(a['image_id']); shutil.copy2(path,CASE/path.name)
        with Image.open(path) as im: ax.imshow(im)
        refs=collections.Counter(x['answer'] for x in a['answers']).most_common(3)
        text=f"Q{qid}: {a['question']}\nGreedy: {g.answer} ({g.score:.0f}) | Beam3: {b.answer} ({b.score:.0f})\nReferences: "+', '.join(f'{v} [{c}/10]' for v,c in refs)
        ax.set_title(textwrap.fill(text.split('\n')[0],65)+'\n'+'\n'.join(text.split('\n')[1:]),fontsize=9,loc='left'); ax.axis('off')
    for ax in axes[len(ids):]: ax.axis('off')
    fig.suptitle(title,fontsize=16); fig.tight_layout(rect=(0,0,1,.96)); savefig(fig,name)
gallery(chosen,'07_error_cases','Zero-credit cases: deterministic selection by answer type')
success=base.query('score==100').groupby('answer_type',sort=True).head(2).question_id.tolist(); gallery(success,'08_success_cases','Full-credit cases: deterministic selection by answer type')
print('SUMMARY'); print(summary.to_string(index=False)); print('TIMING'); print(time_summary); print('PAIRED',changes.to_dict(),ci)
print('ERROR CASES'); print(case_rows[['question_id','question','answer','reference_majority','references']].to_string(index=False))
# 保存训练数据重叠证据与机器可读限制
train_url='https://raw.githubusercontent.com/salesforce/BLIP/main/configs/vqa.yaml'
r=requests.get(train_url,timeout=60); r.raise_for_status(); (OUT/'BLIP_official_vqa_config.yaml').write_text(r.text)
limitations={'evaluation_scope':'fixed 500-question VQAv2 validation diagnostic','training_overlap':'Official BLIP VQA config lists vqa_train, vqa_val and vg_qa; checkpoint-specific lineage not independently proven. Treat as potentially contaminated, not unseen-test generalization.','source':train_url,'no_project_finetuning':True,'timing':'amortized synchronized GPU generation only, batch size 8; no warmup separation','case_selection':'first two zero-credit examples per answer type sorted by question_id; descriptive, not causal attribution'}
(OUT/'limitations.json').write_text(json.dumps(limitations,ensure_ascii=False,indent=2))
report=['# 实验5：结果、图表与报告素材','', '**评测定位：固定 500 题 validation 诊断，可能存在预训练数据重叠，不是独立测试集成绩。**','', '## 方法','模型为 Salesforce/blip-vqa-base，输入图片和问题，生成简短答案；对照 greedy 与 beam=3，其余配置一致。项目未进行微调。','VQA 官方评测器仅作 Python 3 print 兼容修改。逐题对十个人工答案执行留一法，得分可为 0、30、60、90、100；最终取平均。','', '## 结果',summary.to_markdown(index=False),'', '## 对照分析',f'Beam3 - Greedy = {delta.mean():.2f} 个百分点；图像聚类 bootstrap 95% 区间 {ci["delta"]}。若区间包含零，不能据此断言存在稳定差异。',f'逐题改善/不变/退步：{changes.to_dict()}。计数题仅53题，类别差异需要结合样本量理解。','', '## 耗时',time_summary.to_markdown(),'耗时仅为同步后的GPU批量生成摊销时间，不包含下载、图片加载和预处理，也未单独剔除预热批次。','', '## 错误案例',case_rows[['question_id','question','answer','reference_majority']].to_markdown(index=False),'这些是按类别和题号固定选择的零分案例；错误可能与计数、空间关系、语义歧义或答案表述有关，不能仅由预测文本确定内部原因。','', '## 数据泄漏与适用范围','官方 BLIP 配置 train_files 包含 vqa_train、vqa_val、vg_qa。当前结果不能报告为未见数据泛化成绩。后续应换用可核验的独立数据或仅在训练划分训练的模型进行独立测试。',train_url,'', '## 图表使用建议','01：总体及类别结果；02：数据构成；03：计算开销；04：逐题对照；05：问题类型；06：得分分布；07/08：错误与成功案例。PNG 为 240dpi，PDF 为可缩放图形。','', '## PPT 建议（素材提纲）','任务与模型 → 数据与重叠限制 → 评测公式 → 总体/类别图 → 解码成本与配对差异 → 错误案例 → 局限与改进。','', '## 参考资料','BLIP: https://arxiv.org/abs/2201.12086','VQAv2: https://visualqa.org/','Evaluator: https://github.com/GT-Vision-Lab/VQA','Model: https://huggingface.co/Salesforce/blip-vqa-base']
(OUT/'实验5_结果分析与PPT素材.md').write_text('\n\n'.join(report))
print('REPORT AND FIGURES READY',len(list(FIG.glob('*.png'))))


In [ ]:
# 阶段7：案例解释、源码快照、完整性校验与可持久下载成果包
import base64, zipfile, hashlib, copy, json
from IPython.display import HTML, Image as DisplayImage, display
notes='''

## 六个零分案例的解释（根据题目与人工答案，不推断模型内部原因）
- Q1903001：问盖子里有什么，模型答 lid，8/10 人答 nothing。模型复述了问题中的物体名，没有回答盖内内容。
- Q34950010：列车颜色预测 red and blue；人工答案列举的颜色组合高度分散。模型漏报部分颜色，同时精确答案匹配对多色组合及顺序敏感。零分不等于完全没有识别颜色。
- Q50125003：轮胎数预测10；参考7和9各3票、8有2票，另有3和6各1票。这是计数错误与标注歧义并存，不能把7写成唯一确定真值。
- Q71724004：船上人数预测2，人工答案3获得10/10票，是明确的漏计案例。
- Q323951000：帽子问题预测yes，参考no获10/10票，属于物体属性判断错误。
- Q483165001：头盔问题预测yes，参考no获10/10票，属于物体属性判断错误。不能由这两个案例直接断言模型存在总体yes偏置。

图表07的案例按答案类型和question_id确定性选取，仅用于解释，不代表各类错误的发生比例。
'''
p=OUT/'实验5_结果分析与PPT素材.md'; text=p.read_text(); p.write_text(text.split('## 六个零分案例的解释')[0]+notes)
# 每个图均附同一结论边界
(FIG/'图表说明.md').write_text('所有准确率图均为固定500题validation诊断；官方BLIP训练配置含vqa_val，可能存在训练重叠。PNG 240dpi，PDF可缩放。耗时为GPU批量生成摊销时间，不是端到端请求延迟。案例图片来自COCO，仅选择12个问题对应的图像用于复核。')
# 保存当前 notebook 的源码，去除输出以避免递归嵌入ZIP
from google.colab import _message
nb=copy.deepcopy(_message.blocking_request('get_ipynb',timeout_sec=60)['ipynb'])
for cell in nb.get('cells',[]):
    if cell.get('cell_type')=='code': cell['outputs']=[]; cell['execution_count']=None
(OUT/'实验5_可复现源码.ipynb').write_text(json.dumps(nb,ensure_ascii=False,indent=1))
(OUT/'requirements.txt').write_text('transformers==4.57.1\npandas\nnumpy\nmatplotlib\npillow\nrequests\ntabulate\n# PyTorch: see run_config.json; use Colab bundled CUDA build.\n')
(OUT/'README.md').write_text('''# 实验5成果包

先阅读 实验5_结果分析与PPT素材.md 和 limitations.json。
- figures/: 8张PNG + 8张PDF与使用说明。
- per_question_results.csv: 两组共1000行逐题结果。
- error_cases.csv: 六个零分案例；case_images/: 选中案例图片。
- accuracy_summary.csv, timing_summary.csv, paired_statistics.json: 汇总与配对统计。
- predictions_*.jsonl, submission_*.json: 原始预测与标准提交格式。
- eval_manifest.json: 固定500题的题目、图像ID与十人人工答案。
- run_config.json, data_sources.json, evaluator_source.json: 模型/环境/来源。
- 实验5_可复现源码.ipynb: 在Colab T4从上到下运行；图片和模型首次运行需要下载。
- vqaEval.py 与原始版本: 官方评测实现和兼容修改记录。
- BLIP_official_vqa_config.yaml: 官方训练划分含validation的证据。

这不是独立未见测试集成绩。没有进行项目微调。耗时和bootstrap区间均有适用范围，见报告。

源码快照不保存模型权重或全部数据图片；通过清单与官方下载源重建。
''')
assert len(df)==1000 and len(list(FIG.glob('*.png')))==8 and len(list(FIG.glob('*.pdf')))==8
checksum={str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.rglob('*')) if p.is_file() and p.name!='checksums_sha256.json'}
(OUT/'checksums_sha256.json').write_text(json.dumps(checksum,indent=2,ensure_ascii=False))
archive=Path(shutil.make_archive('/content/实验5_视觉问答成果包','zip',OUT))
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    print('ZIP_FILES',len(z.namelist()))
print('ZIP_BYTES',archive.stat().st_size,'SHA256',hashlib.sha256(archive.read_bytes()).hexdigest())
# 将字节嵌入输出，保存notebook后即可脱离运行时下载
href='data:application/zip;base64,'+base64.b64encode(archive.read_bytes()).decode()
display(HTML('<h3>实验5成果包</h3><p>含图表、逐题结果、案例、报告素材和源码。结果存在训练数据重叠风险，请先阅读报告。</p><a download="experiment5_vqa_results.zip" href="'+href+'">下载实验5完整成果包（ZIP）</a>'))
# 显式展示已保存PNG，便于检查实际导出图
for name in ['01_accuracy_by_type','07_error_cases']:
    display(DisplayImage(filename=str(FIG/(name+'.png')),width=1000))


In [ ]:
print([(str(p.relative_to(OUT)),round(p.stat().st_size/1e6,2)) for p in sorted(OUT.rglob('*'),key=lambda x:x.stat().st_size,reverse=True) if p.is_file()][:12])
